In [1]:
!pip install pandas sentence-transformers chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 88.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.2 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.9.0.post0
    Uninstalling python-dateutil-2.9.0.post0:
      Successfully uninstalled python-dateutil-2.9.0.post0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: oauthlib
    Found existing installation: oauthlib 3.3.1
    Uninstalling oauthlib-3.3.1:
      Successfully uninstalled oauthlib-3.3.1
  Attempting uninstall: opentelemetry-semantic-conven

In [1]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
data = pd.read_csv("faqs.csv")

print("FAQ data loaded successfully!")
print("Number of FAQs:", len(data))

data.head()

FAQ data loaded successfully!
Number of FAQs: 15


,question,answer
0,What is Python?,Python is a programming language used for web ...
1,What is Artificial Intelligence?,Artificial Intelligence is a field of computer...
2,What is Machine Learning?,Machine Learning is a branch of artificial int...
3,What is deep learning?,Deep learning is a type of machine learning th...
4,What is GitHub?,GitHub is an online platform used to store cod...


In [3]:
import re

def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    return text


data["question"] = data["question"].apply(clean_text)
data["answer"] = data["answer"].apply(clean_text)

print("Text cleaning completed!")

data.head()

Text cleaning completed!


,question,answer
0,what is python?,python is a programming language used for web ...
1,what is artificial intelligence?,artificial intelligence is a field of computer...
2,what is machine learning?,machine learning is a branch of artificial int...
3,what is deep learning?,deep learning is a type of machine learning th...
4,what is github?,github is an online platform used to store cod...


In [4]:
def split_into_chunks(text, chunk_size=50, overlap=10):
    words = text.split()
    chunks = []

    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# Test the chunking function
sample_text = data["answer"][0]
chunks = split_into_chunks(sample_text)

print("Original text:")
print(sample_text)

print("\nChunks:")
for chunk in chunks:
    print(chunk)

Original text:
python is a programming language used for web development, automation, data analysis, and artificial intelligence.

Chunks:
python is a programming language used for web development, automation, data analysis, and artificial intelligence.


In [5]:
def prepare_documents(data):
    documents = []
    metadatas = []
    ids = []

    for index, row in data.iterrows():

        question = clean_text(row["question"])
        answer = clean_text(row["answer"])

        text = "Question: " + question + " Answer: " + answer

        chunks = split_into_chunks(text)

        for chunk_number, chunk in enumerate(chunks):

            documents.append(chunk)

            metadatas.append({
                "question": question,
                "answer": answer
            })

            ids.append(str(index) + "_" + str(chunk_number))

    return documents, metadatas, ids


documents, metadatas, ids = prepare_documents(data)

print("Documents prepared successfully!")
print("Total documents:", len(documents))
print("\nFirst document:")
print(documents[0])

Documents prepared successfully!
Total documents: 15

First document:
Question: what is python? Answer: python is a programming language used for web development, automation, data analysis, and artificial intelligence.


In [6]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Sentence Transformer model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence Transformer model loaded successfully!


In [7]:
embeddings = model.encode(documents)

print("Embeddings created successfully!")
print("Number of embeddings:", len(embeddings))
print("Embedding size:", embeddings.shape)

Embeddings created successfully!
Number of embeddings: 15
Embedding size: (15, 384)


In [8]:
client = chromadb.PersistentClient(path="chroma_db")

collection = client.get_or_create_collection(
    name="faq_collection",
    metadata={"hnsw:space": "cosine"}
)

print("ChromaDB collection created successfully!")

ChromaDB collection created successfully!


In [9]:
collection.upsert(
    ids=ids,
    documents=documents,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print("Embeddings stored successfully!")
print("Total records in database:", collection.count())

Embeddings stored successfully!
Total records in database: 15


In [10]:
query = input("Enter your question: ")

query = clean_text(query)

query_embedding = model.encode([query]).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=3
)

print("\nTop 3 matching FAQs:")

for i in range(len(results["ids"][0])):

    print("\nResult", i + 1)
    print("Question:", results["metadatas"][0][i]["question"])
    print("Answer:", results["metadatas"][0][i]["answer"])

Enter your question: How can computers learn from information?

Top 3 matching FAQs:

Result 1
Question: what is machine learning?
Answer: machine learning is a branch of artificial intelligence that allows computers to learn patterns from data and make predictions.

Result 2
Question: what is artificial intelligence?
Answer: artificial intelligence is a field of computer science that enables machines to perform tasks that normally require human intelligence.

Result 3
Question: what is deep learning?
Answer: deep learning is a type of machine learning that uses neural networks with multiple layers to learn complex patterns.
